In [1]:
import pandas as pd
import numpy as np

# Load dữ liệu gốc
df = pd.read_csv("salary_survey_raw.csv")

print("--- THÔNG TIN TRƯỚC KHI XỬ LÝ ---")
print(f"Kích thước ban đầu (Shape): {df.shape}")
print(f"Tổng số dòng trùng lặp hoàn toàn: {df.duplicated().sum()}")
print("\nKiểu dữ liệu của các cột (Dtypes):")
print(df.dtypes)

--- THÔNG TIN TRƯỚC KHI XỬ LÝ ---
Kích thước ban đầu (Shape): (2800, 17)
Tổng số dòng trùng lặp hoàn toàn: 38

Kiểu dữ liệu của các cột (Dtypes):
timestamp                          str
how_old_are_you                    str
industry                           str
job_title                          str
additional_context_on_job_title    str
annual_salary                      str
additional_monetary_comp           str
currency                           str
income_context                     str
country                            str
us_state                           str
city                               str
years_of_experience_in_field       str
years_of_experience_overall        str
highest_level_of_education         str
gender                             str
race                               str
dtype: object


In [2]:
def missing_report(df):
    """Hàm thống kê số lượng và tỷ lệ % dữ liệu khuyết thiếu."""
    miss = df.isnull().sum()
    pct = (miss / len(df) * 100).round(2)
    return pd.DataFrame({'count': miss, 'pct': pct}).query('count > 0').sort_values('pct', ascending=False)

# Hiển thị báo cáo khuyết thiếu ban đầu
print(missing_report(df))

                                 count    pct
income_context                    2269  81.04
us_state                          1813  64.75
city                              1562  55.79
additional_monetary_comp          1538  54.93
additional_context_on_job_title    969  34.61
race                               766  27.36
annual_salary                      391  13.96
country                            115   4.11
gender                              69   2.46


In [3]:
# Lưu lại số lượng null trước khi xử lý để làm bảng so sánh
null_before = df.isnull().sum()

# Chiến lược 1: Fillna bằng Median có điều kiện (Theo cặp Industry + Currency) cho cột lương
df['currency_clean'] = df['currency'].astype(str).str.strip().str.upper()
df['annual_salary_clean'] = df['annual_salary'].astype(str).str.replace(r'[$,\s]', '', regex=True)
df['annual_salary_clean'] = pd.to_numeric(df['annual_salary_clean'], errors='coerce')

global_median = df['annual_salary_clean'].median()
df['annual_salary_clean'] = df.groupby(['industry', 'currency_clean'])['annual_salary_clean'].transform(
    lambda x: x.fillna(x.median() if not x.dropna().empty else global_median)
)

# Chiến lược 2: Fillna bằng hằng số 'Unknown' / 'No Context' cho các biến categorical
df['years_of_experience_in_field'] = df['years_of_experience_in_field'].fillna('Unknown')
df['years_of_experience_overall'] = df['years_of_experience_overall'].fillna('Unknown')
df['country'] = df['country'].fillna('Unknown')
df['gender'] = df['gender'].fillna('Unknown')
df['race'] = df['race'].fillna('Unknown')
df['additional_context_on_job_title'] = df['additional_context_on_job_title'].fillna('No Context') 
df['income_context'] = df['income_context'].fillna('No Context')

# Chiến lược 3: Điền giá trị mặc định bằng 0 cho phần tiền thưởng khuyết
df['additional_monetary_comp_clean'] = df['additional_monetary_comp'].astype(str).str.replace(r'[$,\s]', '', regex=True)
df['additional_monetary_comp_clean'] = pd.to_numeric(df['additional_monetary_comp_clean'], errors='coerce').fillna(0)

# Tạo bảng so sánh Null Count Trước và Sau xử lý (Yêu cầu bắt buộc của Checklist 1.1)
null_after = df.isnull().sum()

In [4]:
# Kiểm tra trùng lặp toàn hàng
n_dup = df.duplicated().sum()
print(f"Số hàng trùng lặp toàn bộ: {n_dup} ({n_dup/len(df)*100:.1f}%)") 

# Kiểm tra trùng lặp theo cặp key columns chính (Ví dụ: timestamp và job_title)
key_dup = df.duplicated(subset=['timestamp', 'job_title']).sum() 
print(f"Số hàng trùng lặp theo thuộc tính khóa (timestamp + job_title): {key_dup}")

# Lưu lại số lượng hàng trước khi drop
shape_before_drop = df.shape[0]

# Tiến hành loại bỏ trùng lặp, giữ lại dòng đầu tiên
df = df.drop_duplicates(keep='first').reset_index(drop=True) 
shape_after_drop = df.shape[0]

Số hàng trùng lặp toàn bộ: 38 (1.4%)
Số hàng trùng lặp theo thuộc tính khóa (timestamp + job_title): 107


In [5]:
# Đo lượng bộ nhớ tiêu thụ của cột phân loại 'industry' trước khi convert
mem_before = df['industry'].memory_usage(deep=True)

# 1. Ép kiểu thời gian (Sửa lỗi Mixed Format gây mất mát dữ liệu NaT)
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce', format='mixed') 

# 2. Ép kiểu Category cho cột phân loại để tối ưu hóa bộ nhớ
df['industry'] = df['industry'].astype('category') 

# Đo lượng bộ nhớ tiêu thụ sau khi convert
mem_after = df['industry'].memory_usage(deep=True)
saved_pct = ((mem_before - mem_after) / mem_before) * 100

print(f"Bộ nhớ cột 'industry' TRƯỚC khi tối ưu: {mem_before / 1024:.2f} KB")
print(f"Bộ nhớ cột 'industry' SAU khi tối ưu: {mem_after / 1024:.2f} KB")
print(f"-> Tiết kiệm được: {saved_pct:.2f}% dung lượng bộ nhớ!") 

Bộ nhớ cột 'industry' TRƯỚC khi tối ưu: 189.65 KB
Bộ nhớ cột 'industry' SAU khi tối ưu: 4.46 KB
-> Tiết kiệm được: 97.65% dung lượng bộ nhớ!


In [ ]:
# ==============================================================================
# BẢNG TỔNG KẾT VÀ SO SÁNH TRƯỚC / SAU KHI XỬ LÝ TOÀN BỘ DATASET (PHẦN 1)
# ==============================================================================

# 1. Thu thập các chỉ số sau khi xử lý
null_after = df.isnull().sum()
dtypes_after = df.dtypes

# 2. Tự động tạo bảng so sánh Trước vs Sau cho TOÀN BỘ các cột dữ liệu
summary_columns_df = pd.DataFrame({
    'Cột dữ liệu': df.columns,
    'Kiểu dữ liệu (Trước)': [null_before.index.map(df_orig.dtypes.to_dict()).get(col, 'N/A') for col in df.columns],
    'Kiểu dữ liệu (Sau)': dtypes_after.values,
    'Số lượng Null (Trước)': [null_before.get(col, 0) for col in df.columns],
    'Số lượng Null (Sau)': null_after.values
})

print("==============================================================================")
print("             BẢNG CHẤT LƯỢNG DỮ LIỆU CHI TIẾT THEO CỘT (TRƯỚC vs SAU)         ")
print("==============================================================================")
print(summary_columns_df.to_string(index=False))
print("\n")

# 3. Tạo bảng so sánh các chỉ số tổng quan (Shape & Duplicates) theo đúng Rubric
summary_global_df = pd.DataFrame({
    'Chỉ số tổng quan': ['Tổng số hàng (Rows)', 'Tổng số cột (Columns)', 'Số dòng trùng lặp (Duplicates)'],
    'Trước xử lý': [2800, 17, 38], # Số liệu gốc ban đầu
    'Sau xử lý': [df.shape[0], df.shape[1], df.duplicated().sum()]
})

print("==============================================================================")
print("             BẢNG SO SÁNH CHỈ SỐ TỔNG QUAN HỆ THỐNG (TRƯỚC vs SAU)            ")
print("==============================================================================")
print(summary_global_df.to_string(index=False))

NameError: name 'df_orig' is not defined

: 